# 03 — Process a Synthetic Batch

Load one matched synthetic run, validate its base/commercial/low-cost JSON files, and create consistently named regular, MCMC, and best-Pareto NetCDF outputs. `RUN_ID=None` selects the latest Stage-1 run.

## Setup

In [116]:
import json
import os
import pathlib
import sys
import tempfile

import numpy as np
import pandas as pd
import xarray as xr
try:
    from IPython.display import display
except ImportError:
    display = print

CWD = pathlib.Path.cwd().resolve()
for candidate in [CWD, *CWD.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root containing pyproject.toml.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from soilgasflux_fcs import Multiprocessor, json_reader

RAW_RUNS_ROOT = REPO_ROOT / "notebooks" / "synthetic_create" / "generated" / "03_runs"
PROCESSED_RUNS_ROOT = REPO_ROOT / "notebooks" / "processing" / "output" / "03_synthetic_runs"
SENSOR_ORDER = ["base", "commercial", "low-cost"]


## Configuration

Production execution uses 500 MCMC steps per fit. `SOILGASFLUX_N_MC` can temporarily reduce that value for a smoke test without editing the notebook.

In [117]:
RUN_ID = os.environ.get("SOILGASFLUX_RUN_ID") or None
N_MC = int(os.environ.get("SOILGASFLUX_N_MC", "2000"))
AREA_CM2 = 314.0
VOLUME_CM3 = 6283.0

MCMC_PRECISION_PPM = {
    "base": None,
    "commercial": 10,
    "low-cost": 5,
}


## Run Discovery and Validation

In [118]:
def read_json(path):
    with pathlib.Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def resolve_run(root, run_id, manifest_name="manifest.json"):
    if run_id:
        run_dir = root / run_id
        if not (run_dir / manifest_name).exists():
            raise FileNotFoundError(f"Run {run_id!r} does not contain {manifest_name}: {run_dir}")
        return run_dir
    candidates = sorted(
        path for path in root.glob("*")
        if path.is_dir() and (path / manifest_name).exists()
    )
    if not candidates:
        raise FileNotFoundError(
            f"No generated runs were found in {root}. Run 03_paper_synthetic_batch.ipynb first."
        )
    return candidates[-1]


def validate_raw_manifest(run_dir, manifest):
    if manifest.get("schema_version") != 1:
        raise ValueError(f"Unsupported raw manifest schema: {manifest.get('schema_version')!r}")
    if manifest.get("run_id") != run_dir.name:
        raise ValueError(
            f"Manifest run_id {manifest.get('run_id')!r} does not match folder {run_dir.name!r}."
        )
    scenarios = manifest.get("scenarios", [])
    if len(scenarios) != 6:
        raise ValueError(f"Expected six scenarios, found {len(scenarios)}.")
    expected_ids = {scenario["scenario_id"] for scenario in scenarios}
    if len(expected_ids) != 6:
        raise ValueError("Scenario IDs must be unique.")

    for sensor_type in SENSOR_ORDER:
        folder = run_dir / sensor_type
        files = sorted(folder.glob("*.json"))
        actual_ids = {path.stem for path in files}
        if actual_ids != expected_ids:
            raise ValueError(
                f"{sensor_type!r} scenario mismatch. Expected {sorted(expected_ids)}, found {sorted(actual_ids)}."
            )
        for scenario in scenarios:
            relative = scenario.get("files", {}).get(sensor_type)
            if relative is None or not (run_dir / relative).exists():
                raise FileNotFoundError(
                    f"Missing {sensor_type!r} JSON for scenario {scenario['scenario_id']!r}."
                )

    start_dates = {
        pd.Timestamp(scenario["start_time_utc"]).date()
        for scenario in scenarios
    }
    if len(start_dates) != 1:
        raise ValueError(
            "A synthetic batch must fit within one processing date; found "
            f"{sorted(str(value) for value in start_dates)}."
        )
    return scenarios


raw_run_dir = resolve_run(RAW_RUNS_ROOT, RUN_ID)
raw_manifest = read_json(raw_run_dir / "manifest.json")
scenarios = validate_raw_manifest(raw_run_dir, raw_manifest)
run_id = raw_manifest["run_id"]
processed_run_dir = PROCESSED_RUNS_ROOT / run_id
processed_run_dir.mkdir(parents=True, exist_ok=True)

print(f"Selected raw run: {raw_run_dir}")
print(f"Processed outputs: {processed_run_dir}")


Selected raw run: /Users/alexnaokiasatokobayashi/git/soilgasflux_fcs/notebooks/synthetic_create/generated/03_runs/20260805T102850365113Z
Processed outputs: /Users/alexnaokiasatokobayashi/git/soilgasflux_fcs/notebooks/processing/output/03_synthetic_runs/20260805T102850365113Z


## Processing Helpers

In [119]:
def load_sensor_dataframe(run_dir, sensor_type):
    folder = run_dir / sensor_type
    initializer = json_reader.Initializer(folderPath=folder)
    df = initializer.prepare_rawdata()
    expected_ids = {scenario["scenario_id"] for scenario in scenarios}
    actual_ids = set(df["id"].astype(str).unique())
    if actual_ids != expected_ids:
        raise ValueError(
            f"Loaded IDs for {sensor_type!r} do not match the manifest. "
            f"Expected {sorted(expected_ids)}, found {sorted(actual_ids)}."
        )
    required = {
        "datetime",
        "id",
        "timedelta",
        "k30_co2",
        "si_temperature",
        "si_humidity",
        "bmp_pressure",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required processing columns for {sensor_type!r}: {sorted(missing)}")
    if df[list(required)].isna().any().any():
        raise ValueError(f"Null values found in required processing columns for {sensor_type!r}.")
    return df


def enrich_dataset(ds, manifest, sensor_type, processing_kind, mcmc_precision):
    required_coords = {"time"}
    if processing_kind != "best_pareto":
        required_coords.update({"cutoff", "deadband"})
    missing_coords = required_coords - set(ds.coords)
    if missing_coords:
        raise ValueError(
            f"{processing_kind} dataset is missing coordinates: {sorted(missing_coords)}"
        )

    scenario_by_time = {
        pd.Timestamp(scenario["start_time_utc"]): scenario
        for scenario in manifest["scenarios"]
    }
    matched = []
    for value in ds["time"].values:
        key = pd.Timestamp(value)
        if key not in scenario_by_time:
            raise ValueError(f"Processed time {key} is not present in the raw manifest.")
        matched.append(scenario_by_time[key])

    enriched = ds.assign_coords(
        scenario_id=("time", [item["scenario_id"] for item in matched]),
        curve_type=("time", [item["curve_type"] for item in matched]),
        intensity=("time", [item["intensity"] for item in matched]),
        alpha=("time", [item["ideal_parameters"]["alpha"] for item in matched]),
        c_s=("time", [item["ideal_parameters"]["c_s"] for item in matched]),
        c0=("time", [item["ideal_parameters"]["c0"] for item in matched]),
    )
    enriched.attrs.update(
        {
            "schema_version": 1,
            "run_id": manifest["run_id"],
            "sensor_type": sensor_type,
            "processing_kind": processing_kind,
            "mcmc_precision_ppm": (
                "package_default" if mcmc_precision is None else float(mcmc_precision)
            ),
            "source_manifest": str(raw_run_dir / "manifest.json"),
        }
    )
    return enriched


def validate_processed_dataset(ds, processing_kind):
    required_coords = {"time", "cutoff", "deadband", "scenario_id", "curve_type", "intensity"}
    missing_coords = required_coords - set(ds.coords)
    if missing_coords:
        raise ValueError(f"{processing_kind} output is missing coordinates: {sorted(missing_coords)}")
    required_vars = {"dcdt(HM)"}
    if processing_kind == "mcmc":
        required_coords.add("MC")
        required_vars.add("logprob(HM)")
    missing_vars = required_vars - set(ds.data_vars)
    if missing_vars:
        raise ValueError(f"{processing_kind} output is missing variables: {sorted(missing_vars)}")
    if processing_kind == "mcmc" and "MC" not in ds.coords:
        raise ValueError("MCMC output is missing its MC coordinate.")
    if ds.sizes.get("time") != 6:
        raise ValueError(f"{processing_kind} output contains {ds.sizes.get('time')} scenarios instead of six.")


def save_dataset(ds, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_netcdf(path)
    if not path.exists() or path.stat().st_size == 0:
        raise IOError(f"NetCDF output was not written correctly: {path}")


## Process Base, Commercial, and Low-Cost Data

In [120]:
metadata = {"area": AREA_CM2, "volume": VOLUME_CM3}
processing_manifest = {
    "schema_version": 1,
    "run_id": run_id,
    "source_manifest": str(raw_run_dir / "manifest.json"),
    "n_mc": N_MC,
    "metadata": metadata,
    "sensor_order": SENSOR_ORDER,
    "sensors": {},
}
datasets = {"regular": {}, "mcmc": {}, "best_pareto": {}}

for sensor_type in SENSOR_ORDER:
    print()
    print(f"Processing {sensor_type}...")
    df = load_sensor_dataframe(raw_run_dir, sensor_type)
    processor = Multiprocessor()
    precision = MCMC_PRECISION_PPM[sensor_type]

    with tempfile.TemporaryDirectory(prefix=f"soilgasflux-{sensor_type}-") as temporary_output:
        regular_ds = processor.run(
            df=df,
            chamber_id=f"{sensor_type}_regular",
            output_folder=temporary_output,
            metadata=metadata,
        )
        mcmc_ds = processor.run_MC(
            df=df,
            chamber_id=f"{sensor_type}_mcmc",
            output_folder=temporary_output,
            save_netcdf=False,
            sensor_precision=precision,
            n_MC=N_MC,
            metadata=metadata,
        )

    regular_ds = enrich_dataset(
        regular_ds,
        raw_manifest,
        sensor_type,
        processing_kind="regular",
        mcmc_precision=precision,
    )
    mcmc_ds = enrich_dataset(
        mcmc_ds,
        raw_manifest,
        sensor_type,
        processing_kind="mcmc",
        mcmc_precision=precision,
    )
    validate_processed_dataset(regular_ds, "regular")
    validate_processed_dataset(mcmc_ds, "mcmc")

    best_ds = processor.select_bestPareto(
        ds=mcmc_ds,
        chamber_id=f"{sensor_type}_mcmc",
        date=pd.Timestamp(mcmc_ds["time"].values[0]).date(),
        output_folder=None,
    )
    best_ds = enrich_dataset(
        best_ds,
        raw_manifest,
        sensor_type,
        processing_kind="best_pareto",
        mcmc_precision=precision,
    )

    regular_name = f"{sensor_type}_regular.nc"
    mcmc_name = f"{sensor_type}_mcmc.nc"
    best_name = f"{sensor_type}_mcmc_best_pareto.nc"
    save_dataset(regular_ds, processed_run_dir / regular_name)
    save_dataset(mcmc_ds, processed_run_dir / mcmc_name)
    save_dataset(best_ds, processed_run_dir / best_name)

    datasets["regular"][sensor_type] = regular_ds
    datasets["mcmc"][sensor_type] = mcmc_ds
    datasets["best_pareto"][sensor_type] = best_ds
    processing_manifest["sensors"][sensor_type] = {
        "input_folder": str(raw_run_dir / sensor_type),
        "mcmc_precision_ppm": "package_default" if precision is None else precision,
        "regular": regular_name,
        "mcmc": mcmc_name,
        "best_pareto": best_name,
    }

with (processed_run_dir / "processing_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(processing_manifest, handle, indent=2)

expected_times = datasets["regular"]["base"]["time"].values
for kind in ["regular", "mcmc"]:
    for sensor_type in SENSOR_ORDER:
        if not np.array_equal(datasets[kind][sensor_type]["time"].values, expected_times):
            raise ValueError(f"Time coordinates are not aligned for {kind}/{sensor_type}.")

output_summary = pd.DataFrame(
    [
        {
            "sensor_type": sensor_type,
            "regular": processing_manifest["sensors"][sensor_type]["regular"],
            "mcmc": processing_manifest["sensors"][sensor_type]["mcmc"],
            "best_pareto": processing_manifest["sensors"][sensor_type]["best_pareto"],
            "precision_ppm": processing_manifest["sensors"][sensor_type]["mcmc_precision_ppm"],
        }
        for sensor_type in SENSOR_ORDER
    ]
)
display(output_summary)
print(f"Processing manifest: {processed_run_dir / 'processing_manifest.json'}")
print(f"Next: run notebooks/post_processing/03_pareto_lowcost_vs_commercial.ipynb with RUN_ID={run_id!r}, or leave RUN_ID=None to select this latest run.")



Processing base...

Processing commercial...

Processing low-cost...


,sensor_type,regular,mcmc,best_pareto,precision_ppm
0,base,base_regular.nc,base_mcmc.nc,base_mcmc_best_pareto.nc,package_default
1,commercial,commercial_regular.nc,commercial_mcmc.nc,commercial_mcmc_best_pareto.nc,10
2,low-cost,low-cost_regular.nc,low-cost_mcmc.nc,low-cost_mcmc_best_pareto.nc,5


Processing manifest: /Users/alexnaokiasatokobayashi/git/soilgasflux_fcs/notebooks/processing/output/03_synthetic_runs/20260805T102850365113Z/processing_manifest.json
Next: run notebooks/post_processing/03_pareto_lowcost_vs_commercial.ipynb with RUN_ID='20260805T102850365113Z', or leave RUN_ID=None to select this latest run.
